# Evaluating Models

## What you'll learn
- The confusion matrix: the four ways a yes/no prediction can land
- Precision, recall, and F1, and when each matters
- ROC AUC: how well a model ranks accounts by risk
- Cross-validation: a more reliable score than one train/test split
- How to compare models fairly

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values
print(df.shape)

In [ ]:
NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X = df[NUMERIC]
y = df["paid_late"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
predicted = model.predict(X_test)

## The confusion matrix

| | Predicted on time | Predicted late |
|---|---|---|
| **Actually on time** | True negative | False positive (false alarm) |
| **Actually late** | False negative (missed) | True positive (caught) |

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, predicted, display_labels=["on time", "late"])
plt.title("Late payment: logistic regression")
plt.show()

## Precision, recall, F1

- **Precision:** of the accounts we flagged, how many were really late? (Few false alarms.)
- **Recall:** of the accounts that were really late, how many did we flag? (Few misses.)
- **F1:** one number that balances the two.

`classification_report` prints them all for each class.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predicted, target_names=["on time", "late"]))

## ROC AUC: ranking quality

Precision and recall depend on the threshold. **ROC AUC** looks at every threshold at once
and answers: if you pick one late account and one on-time account at random, how often
does the model give the late one a higher risk score? 0.5 is a coin toss; 1.0 is perfect.

In [ ]:
from sklearn.metrics import roc_auc_score, RocCurveDisplay

probability = model.predict_proba(X_test)[:, 1]
print(f"ROC AUC: {roc_auc_score(y_test, probability):.3f}")
RocCurveDisplay.from_predictions(y_test, probability)
plt.plot([0, 1], [0, 1], color="grey", linestyle="--")
plt.show()

## Cross-validation

A single split can be lucky or unlucky. **Cross-validation** splits the data into, say, 5
parts (folds), trains 5 times each with a different fold held out, and reports all 5
scores. The average is a more reliable estimate, and the spread shows how stable it is.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()),
                         X, y, cv=5, scoring="roc_auc")
print("fold scores:", scores.round(3))
print(f"mean {scores.mean():.3f}, spread (std) {scores.std():.3f}")

## Comparing models fairly

Compare models on the **same** folds and the **same** metric, chosen for the business
problem. Here ROC AUC fits, because the team will work through a ranked list of risky
accounts.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
candidates = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "decision tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=42),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            random_state=42, n_jobs=-1),
}
for name, candidate in candidates.items():
    s = cross_val_score(candidate, X, y, cv=folds, scoring="roc_auc")
    print(f"{name:25s} ROC AUC {s.mean():.3f} (+/- {s.std():.3f})")

## Practice

1. From the confusion matrix, compute precision and recall by hand and check them
   against `classification_report`.
2. Re-run the model comparison using `scoring="recall"`. Does the ranking change?
3. Run cross-validation with `cv=10`. Is the spread smaller or larger than with 5 folds?
4. Using `cross_val_score` and `scoring="neg_mean_absolute_error"`, compare linear
   regression and a random forest for predicting `next_year_spend`.

In [ ]:
# Your practice code here

## Summary

- The confusion matrix shows hits, misses, and false alarms.
- Precision limits false alarms; recall limits misses; F1 balances them.
- ROC AUC measures how well the model ranks cases, across all thresholds.
- Cross-validation gives a more reliable score than a single split.
- Compare models on the same folds and a metric that fits the business need.

## Practice Solutions

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values

NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import confusion_matrix

X, y = df[NUMERIC], df["paid_late"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)

# 1.
tn, fp, fn, tp = confusion_matrix(y_test, model.predict(X_test)).ravel()
print(f"precision {tp / (tp + fp):.2f}, recall {tp / (tp + fn):.2f}")

# 2.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, m in {"logistic": make_pipeline(StandardScaler(), LogisticRegression()),
                "tree": DecisionTreeClassifier(max_depth=4, random_state=42),
                "forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                 random_state=42, n_jobs=-1)}.items():
    print(name, f"recall {cross_val_score(m, X, y, cv=folds, scoring='recall').mean():.3f}")

# 3.
for k in [5, 10]:
    s = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), X, y, cv=k, scoring="roc_auc")
    print(k, "folds: std", round(s.std(), 3))

# 4.
yr = df["next_year_spend"]
for name, m in [("linear", LinearRegression()),
                ("forest", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))]:
    s = -cross_val_score(m, X, yr, cv=5, scoring="neg_mean_absolute_error")
    print(name, f"MAE {s.mean():,.0f}")